[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/053.%20The%20EOQ%20with%20Planned%20Shortages%20Problem/P53-Tier-2_executed.ipynb)

---

# 53. The EOQ with Planned Shortages Problem

## Tier 2: A Greedy Construction Algorithm

This notebook applies a local, greedy search to the economic order quantity (EOQ) model with planned shortages. The order quantity is restricted to integer multiples of a specified batch size. The shortage level remains continuous, as in the model defined below.

The notebook's inputs are annual demand, ordering cost, annual holding cost, annual shortage cost, and the required order multiple. Its outputs include a feasible order quantity and shortage level, their annual cost, the search history, and comparisons with the continuous EOQ solution. The greedy result is a heuristic solution: it is not guaranteed to be the best solution among all feasible discrete order quantities.

### Model assumptions and scope

- Demand is known, constant, and expressed in units per year. Costs are constant over time.
- Replenishment is modeled using the standard EOQ cycle: orders arrive as a batch, and shortages are planned and fully backlogged until replenishment.
- Holding cost is charged per unit of average on-hand inventory per year. Shortage cost is charged per unit of average backlog per year.
- The order quantity must be a positive multiple of `order_multiple`. The maximum shortage, `S`, is a real-valued decision variable between zero and the order quantity.
- The model does not include lead time, safety stock, lost sales, quantity discounts, stochastic demand, or multiple products.

### Example data

The example uses the Meridian Electronics Distribution Center parameters:

| Parameter | Meaning | Value |
|:--|:--|--:|
| `D` | Annual demand | 36,000 units/year |
| `K` | Fixed ordering cost | $150/order |
| `h` | Annual holding cost | $8/unit/year |
| `p` | Annual shortage cost | $12/unit/year |
| `order_multiple` | Required order-quantity multiple | 50 units |

### Learning goals

The steps below define the cost model, construct and improve a feasible solution, compare the heuristic with a continuous lower bound, and examine the search under different starting points and order multiples. The additional experiments illustrate this specific heuristic; they do not establish a general runtime guarantee or an optimality guarantee.

## 1. Cost model and continuous reference solution

Let `Q` be the order quantity and `S` the maximum backlog in one replenishment cycle. When inventory decreases linearly from `Q - S` to zero, and backlog increases linearly from zero to `S`, the annual cost is

$$C(Q,S)=\frac{DK}{Q}+\frac{h(Q-S)^2}{2Q}+\frac{pS^2}{2Q},\qquad Q>0,\quad 0\leq S\leq Q.$$

Here, `D` is annual demand, `K` is the fixed cost per order, `h` is annual holding cost per unit, and `p` is annual shortage cost per unit. The first term is the number of orders per year, `D/Q`, multiplied by `K`. The second and third terms are the annual holding and shortage costs derived from the average inventory and backlog over the cycle. Each term has units of currency per year.

For the continuous reference, both `Q` and `S` may take any feasible real values. Minimizing this cost gives

$$Q^*=\sqrt{\frac{2DK(h+p)}{hp}},\qquad S^*=Q^*\frac{h}{h+p}.$$

These equations provide the continuous optimum, not the discrete-constrained optimum. Since the continuous feasible set includes every feasible discrete order quantity, its cost is a lower bound for the discrete problem. The comparison cell uses this reference to calculate a cost gap; the gap does not prove that the greedy solution is optimal among discrete choices.

In [ ]:
# Import the libraries used for the search, analysis, and visualizations.
import time
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for the EOQ with planned shortages.")

## 2. Greedy search implementation

A **greedy local search** starts from a feasible solution and accepts a tested neighboring move only when that move reduces cost. Here, a neighbor is produced by changing `Q` or `S` by one of the prescribed step sizes. This strategy is useful for constructing a practical solution without enumerating every possible order quantity. It can stop at a local solution because it tests only selected moves and does not explore every feasible pair.

At each iteration, the search first tests increases and decreases in `Q`, holding the shortage level no higher than the new `Q`. If none of those moves improves the cost, it tests increases and decreases in `S`. It stops when an iteration finds no improving move or when the iteration limit is reached. The continuous solution initializes the default search; a supplied starting point is used only in the robustness experiment.

If `I` iterations are performed, this implementation tests a fixed number of candidate moves per iteration, so the search takes `O(I)` cost evaluations and stores `O(I)` history entries. The iteration limit bounds the work, but does not make the result globally optimal.

In [ ]:
class GreedyEOQWithShortages:
    """Greedy local search for EOQ with planned, fully backlogged shortages."""

    def __init__(
        self,
        demand: float,
        ordering_cost: float,
        holding_cost: float,
        shortage_cost: float,
        order_multiple: int = 1,
    ):
        """Store model parameters and check that they define a valid instance."""
        if demand <= 0 or ordering_cost <= 0:
            raise ValueError("Demand and ordering cost must be positive.")
        if holding_cost <= 0 or shortage_cost <= 0:
            raise ValueError("Holding and shortage costs must be positive.")
        if int(order_multiple) != order_multiple or order_multiple <= 0:
            raise ValueError("The order multiple must be a positive integer.")

        self.D = demand
        self.K = ordering_cost
        self.h = holding_cost
        self.p = shortage_cost
        self.order_multiple = int(order_multiple)

    def calculate_total_cost(self, Q: float, S: float) -> float:
        """Return annual cost for a feasible order quantity and shortage level."""
        if Q <= 0 or S < 0 or S > Q:
            return float("inf")

        ordering_cost = self.D * self.K / Q
        holding_cost = self.h * (Q - S) ** 2 / (2 * Q)
        shortage_cost = self.p * S ** 2 / (2 * Q)
        return ordering_cost + holding_cost + shortage_cost

    def get_optimal_continuous_solution(self) -> Tuple[float, float]:
        """Return the analytical optimum when the order quantity is continuous."""
        Q_optimal = np.sqrt(
            2 * self.D * self.K * (self.h + self.p) / (self.h * self.p)
        )
        S_optimal = Q_optimal * self.h / (self.h + self.p)
        return Q_optimal, S_optimal

    def round_to_multiple(self, value: float) -> int:
        """Round a value to the nearest permitted order-quantity multiple."""
        return int(round(value / self.order_multiple) * self.order_multiple)

    def greedy_search(
        self,
        max_iterations: int = 1000,
        tolerance: float = 1e-6,
        verbose: bool = False,
        initial_Q: float = None,
        initial_S: float = None,
    ) -> Dict:
        """Improve a feasible starting point using the prescribed local moves."""
        if max_iterations < 0:
            raise ValueError("The iteration limit must be nonnegative.")
        if tolerance < 0:
            raise ValueError("The improvement tolerance must be nonnegative.")

        if initial_Q is None:
            Q_continuous, S_continuous = self.get_optimal_continuous_solution()
            Q_current = self.round_to_multiple(Q_continuous)
            S_current = S_continuous if initial_S is None else initial_S
        else:
            Q_current = self.round_to_multiple(initial_Q)
            S_current = 0.4 * Q_current if initial_S is None else initial_S

        if Q_current <= 0:
            Q_current = self.order_multiple
        S_current = min(max(S_current, 0), Q_current)
        current_cost = self.calculate_total_cost(Q_current, S_current)

        history = [{
            "iteration": 0,
            "Q": Q_current,
            "S": S_current,
            "cost": current_cost,
        }]
        converged = False
        iteration = 0
        step_sizes = [
            self.order_multiple,
            2 * self.order_multiple,
            5 * self.order_multiple,
        ]

        for iteration in range(1, max_iterations + 1):
            improved = False

            # Test order-quantity moves first, preserving feasibility of S.
            for step_size in step_sizes:
                Q_new = Q_current + step_size
                S_new = min(S_current, Q_new)
                new_cost = self.calculate_total_cost(Q_new, S_new)

                if new_cost < current_cost - tolerance:
                    Q_current, S_current = Q_new, S_new
                    current_cost = new_cost
                    improved = True
                    break

                if Q_current - step_size > 0:
                    Q_new = Q_current - step_size
                    S_new = min(S_current, Q_new)
                    new_cost = self.calculate_total_cost(Q_new, S_new)

                    if new_cost < current_cost - tolerance:
                        Q_current, S_current = Q_new, S_new
                        current_cost = new_cost
                        improved = True
                        break

            # Adjust S only when no tested Q move reduces the cost.
            if not improved:
                for step_size in step_sizes:
                    S_new = min(S_current + step_size, Q_current)
                    new_cost = self.calculate_total_cost(Q_current, S_new)

                    if new_cost < current_cost - tolerance:
                        S_current = S_new
                        current_cost = new_cost
                        improved = True
                        break

                    if S_current - step_size >= 0:
                        S_new = max(0, S_current - step_size)
                        new_cost = self.calculate_total_cost(Q_current, S_new)

                        if new_cost < current_cost - tolerance:
                            S_current = S_new
                            current_cost = new_cost
                            improved = True
                            break

            history.append({
                "iteration": iteration,
                "Q": Q_current,
                "S": S_current,
                "cost": current_cost,
            })

            if verbose and iteration % 100 == 0:
                print(
                    f"Iteration {iteration}: Q={Q_current}, "
                    f"S={S_current:.2f}, annual cost=${current_cost:.2f}"
                )

            if not improved:
                converged = True
                break

        return {
            "Q_optimal": Q_current,
            "S_optimal": S_current,
            "total_cost": current_cost,
            "iterations": iteration,
            "converged": converged,
            "history": history,
        }

print("Greedy EOQ with planned shortages class defined successfully.")

The search keeps `Q` on the permitted multiple grid. The shortage level `S` is not constrained to that grid; it only has to satisfy `0 ≤ S ≤ Q`. `converged` is `True` when the search tests a full iteration and finds no improving move. If the iteration limit is reached while improving moves remain, it is `False`. The recorded history includes the initial solution and every completed iteration.

In [ ]:
# Initialize the solver with the example's stated parameters.
greedy_eoq = GreedyEOQWithShortages(
    demand=36000,
    ordering_cost=150,
    holding_cost=8,
    shortage_cost=12,
    order_multiple=50,
)

print("=" * 60)
print("GREEDY EOQ WITH PLANNED SHORTAGES")
print("=" * 60)
print("Running the local search with order quantities in multiples of 50...")

start_time = time.perf_counter()
greedy_solution = greedy_eoq.greedy_search(verbose=True)
end_time = time.perf_counter()

print(f"\nElapsed time: {end_time - start_time:.6f} seconds")
print(f"Stopped after a no-improvement iteration: {greedy_solution['converged']}")
print(f"Iterations: {greedy_solution['iterations']}")
print(f"Order quantity: {greedy_solution['Q_optimal']:,.0f} units")
print(f"Maximum planned backlog: {greedy_solution['S_optimal']:,.2f} units")
print(f"Annual cost: ${greedy_solution['total_cost']:,.2f}")
print(
    "Demand immediately supplied from on-hand stock: "
    f"{(1 - greedy_solution['S_optimal'] / greedy_solution['Q_optimal']) * 100:.1f}%"
)
print("=" * 60)

assert greedy_solution["Q_optimal"] % greedy_eoq.order_multiple == 0
assert 0 <= greedy_solution["S_optimal"] <= greedy_solution["Q_optimal"]
assert np.isclose(
    greedy_solution["total_cost"],
    greedy_eoq.calculate_total_cost(
        greedy_solution["Q_optimal"], greedy_solution["S_optimal"]
    ),
)

The reported stock-supply percentage is `1 - S/Q`: the fraction of demand in a cycle supplied from on-hand inventory rather than from the planned backlog. It is a model-based fraction, not a probability of avoiding a stockout. The assertions check the order-multiple constraint, the bounds on `S`, and consistency between the reported cost and the cost function.

In [ ]:
def compare_greedy_vs_optimal(greedy_solver: GreedyEOQWithShortages) -> Dict:
    """Compare the heuristic with the continuous analytical lower bound."""
    Q_opt_cont, S_opt_cont = greedy_solver.get_optimal_continuous_solution()
    optimal_cost = greedy_solver.calculate_total_cost(Q_opt_cont, S_opt_cont)

    start = time.perf_counter()
    greedy_solution = greedy_solver.greedy_search()
    greedy_time_ms = (time.perf_counter() - start) * 1000

    cost_gap = greedy_solution["total_cost"] - optimal_cost
    percentage_gap = cost_gap / optimal_cost * 100
    continuous_Q_is_feasible = np.isclose(
        Q_opt_cont / greedy_solver.order_multiple,
        round(Q_opt_cont / greedy_solver.order_multiple),
    )

    comparison_df = pd.DataFrame({
        "Reference": ["Continuous analytical optimum", "Greedy local search"],
        "Order quantity Q (units)": [Q_opt_cont, greedy_solution["Q_optimal"]],
        "Shortage level S (units)": [S_opt_cont, greedy_solution["S_optimal"]],
        "Annual cost ($/year)": [optimal_cost, greedy_solution["total_cost"]],
        "Q satisfies order multiple": [
            continuous_Q_is_feasible,
            greedy_solution["Q_optimal"] % greedy_solver.order_multiple == 0,
        ],
    })

    print("=" * 88)
    print("GREEDY SEARCH AND CONTINUOUS LOWER-BOUND COMPARISON")
    print("=" * 88)
    print(comparison_df.round(3).to_string(index=False))
    print(f"\nCost gap above the continuous lower bound: ${cost_gap:.4f}/year")
    print(f"Percentage gap above the continuous lower bound: {percentage_gap:.4f}%")
    print(f"Greedy-search elapsed time: {greedy_time_ms:.4f} ms")
    print(
        "The continuous reference is a lower bound; this comparison does not "
        "certify discrete optimality."
    )
    print("=" * 88)

    return {
        "comparison_df": comparison_df,
        "cost_gap": cost_gap,
        "percentage_gap": percentage_gap,
        "optimal_solution": (Q_opt_cont, S_opt_cont, optimal_cost),
        "greedy_solution": greedy_solution,
        "greedy_time_ms": greedy_time_ms,
    }

comparison_results = compare_greedy_vs_optimal(greedy_eoq)

A positive percentage gap measures how much the heuristic cost exceeds the continuous optimum's cost. Because the continuous order quantity may not satisfy the required multiple, the continuous reference is generally not a feasible answer to the discrete problem. A small gap is useful evidence about cost quality for this instance, but it is not an optimality certificate.

In [ ]:
def plot_greedy_convergence(greedy_solution: Dict) -> None:
    """Plot the recorded cost and decision trajectory of a greedy search."""
    history = greedy_solution["history"]
    iterations = [entry["iteration"] for entry in history]
    Q_values = [entry["Q"] for entry in history]
    S_values = [entry["S"] for entry in history]
    costs = [entry["cost"] for entry in history]

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Greedy Search History", fontsize=16, fontweight="bold")

    ax1.plot(iterations, costs, color="blue", marker="o", linewidth=2)
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Annual cost ($/year)")
    ax1.set_title("Cost by iteration")
    ax1.grid(True, alpha=0.3)

    ax2.plot(iterations, Q_values, color="green", marker="s", linewidth=2)
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Order quantity Q (units)")
    ax2.set_title("Order quantity by iteration")
    ax2.grid(True, alpha=0.3)

    ax3.plot(iterations, S_values, color="red", marker="^", linewidth=2)
    ax3.set_xlabel("Iteration")
    ax3.set_ylabel("Shortage level S (units)")
    ax3.set_title("Shortage level by iteration")
    ax3.grid(True, alpha=0.3)

    ax4.plot(Q_values, S_values, "mo-", linewidth=2, label="Search path")
    ax4.scatter(Q_values[0], S_values[0], color="green", s=90, label="Start", zorder=5)
    ax4.scatter(Q_values[-1], S_values[-1], color="red", marker="*", s=140, label="End", zorder=5)
    ax4.set_xlabel("Order quantity Q (units)")
    ax4.set_ylabel("Shortage level S (units)")
    ax4.set_title("Search path in Q-S space")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    initial_cost = costs[0]
    final_cost = costs[-1]
    improvement = initial_cost - final_cost
    improvement_percentage = improvement / initial_cost * 100
    print("=" * 60)
    print("SEARCH HISTORY SUMMARY")
    print(f"Initial annual cost: ${initial_cost:,.2f}")
    print(f"Final annual cost: ${final_cost:,.2f}")
    print(f"Cost reduction: ${improvement:,.2f} ({improvement_percentage:.3f}%)")
    print(f"Completed iterations: {len(iterations) - 1}")
    print("=" * 60)

plot_greedy_convergence(greedy_solution)

Each point in the history is the accepted solution after an iteration; rejected candidates are not recorded. Therefore, a flat final segment indicates that the tested moves did not improve the cost. It does not imply that every feasible decision pair has been examined.

In [ ]:
def test_robustness(greedy_solver: GreedyEOQWithShortages) -> pd.DataFrame:
    """Run the same greedy search from several specified initial quantities."""
    starting_points = [100, 500, 1000, 2000, 3000, 5000]
    results = []

    for start_Q in starting_points:
        result = greedy_solver.greedy_search(
            initial_Q=start_Q,
            initial_S=0.4 * start_Q,
        )
        results.append({
            "Starting Q (units)": start_Q,
            "Final Q (units)": result["Q_optimal"],
            "Final S (units)": result["S_optimal"],
            "Final annual cost ($/year)": result["total_cost"],
            "Iterations": result["iterations"],
            "No-improvement stop": result["converged"],
        })

    robustness_df = pd.DataFrame(results)
    print("=" * 92)
    print("STARTING-POINT SENSITIVITY")
    print(robustness_df.round(3).to_string(index=False))

    cost_std = robustness_df["Final annual cost ($/year)"].std()
    cost_range = (
        robustness_df["Final annual cost ($/year)"].max()
        - robustness_df["Final annual cost ($/year)"].min()
    )
    unique_solutions = robustness_df[
        ["Final Q (units)", "Final S (units)", "Final annual cost ($/year)"]
    ].drop_duplicates().shape[0]
    print(f"\nDistinct final solutions: {unique_solutions}")
    print(f"Standard deviation of final annual cost: ${cost_std:.4f}")
    print(f"Range of final annual cost: ${cost_range:.4f}")
    print("Different outcomes indicate sensitivity to the starting point for this search.")
    print("=" * 92)
    return robustness_df

robustness_results = test_robustness(greedy_eoq)

This experiment changes the initial point while keeping the instance and move rules fixed. Comparing final solutions reveals whether these particular starts lead to the same result. It is a finite sensitivity check, not a proof of robustness for all possible starting points.

In [ ]:
def scalability_analysis() -> pd.DataFrame:
    """Measure this greedy search on the stated set of demand instances."""
    test_cases = [
        {"demand": 1000, "K": 50, "h": 2, "p": 3, "name": "Small"},
        {"demand": 10000, "K": 100, "h": 5, "p": 8, "name": "Medium"},
        {"demand": 36000, "K": 150, "h": 8, "p": 12, "name": "Large (base parameters)"},
        {"demand": 100000, "K": 200, "h": 10, "p": 15, "name": "Very large"},
        {"demand": 500000, "K": 500, "h": 20, "p": 30, "name": "Enterprise"},
    ]
    results = []

    for case in test_cases:
        solver = GreedyEOQWithShortages(
            demand=case["demand"],
            ordering_cost=case["K"],
            holding_cost=case["h"],
            shortage_cost=case["p"],
            order_multiple=50,
        )

        start = time.perf_counter()
        solution = solver.greedy_search()
        elapsed_ms = (time.perf_counter() - start) * 1000

        Q_cont, S_cont = solver.get_optimal_continuous_solution()
        continuous_cost = solver.calculate_total_cost(Q_cont, S_cont)
        gap_percentage = (
            (solution["total_cost"] - continuous_cost) / continuous_cost * 100
        )
        results.append({
            "Instance": case["name"],
            "Demand (units/year)": case["demand"],
            "Greedy Q (units)": solution["Q_optimal"],
            "Greedy S (units)": solution["S_optimal"],
            "Greedy cost ($/year)": solution["total_cost"],
            "Continuous lower bound ($/year)": continuous_cost,
            "Gap above lower bound (%)": gap_percentage,
            "Iterations": solution["iterations"],
            "Elapsed time (ms)": elapsed_ms,
        })

    scalability_df = pd.DataFrame(results)
    print("=" * 120)
    print("SEARCH MEASUREMENTS ACROSS THE SPECIFIED INSTANCES")
    print(scalability_df.round(3).to_string(index=False))
    print("=" * 120)

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Greedy Search Measurements by Annual Demand", fontsize=16, fontweight="bold")

    ax1.semilogx(
        scalability_df["Demand (units/year)"],
        scalability_df["Gap above lower bound (%)"],
        "bo-",
        linewidth=2,
        markersize=8,
    )
    ax1.set_xlabel("Annual demand (units/year, logarithmic scale)")
    ax1.set_ylabel("Gap above continuous lower bound (%)")
    ax1.set_title("Cost gap by demand")
    ax1.grid(True, alpha=0.3)

    ax2.semilogx(
        scalability_df["Demand (units/year)"],
        scalability_df["Elapsed time (ms)"],
        "go-",
        linewidth=2,
        markersize=8,
    )
    ax2.set_xlabel("Annual demand (units/year, logarithmic scale)")
    ax2.set_ylabel("Elapsed time (ms)")
    ax2.set_title("Measured elapsed time by demand")
    ax2.grid(True, alpha=0.3)

    ax3.semilogx(
        scalability_df["Demand (units/year)"],
        scalability_df["Iterations"],
        "ro-",
        linewidth=2,
        markersize=8,
    )
    ax3.set_xlabel("Annual demand (units/year, logarithmic scale)")
    ax3.set_ylabel("Iterations")
    ax3.set_title("Iterations by demand")
    ax3.grid(True, alpha=0.3)

    ax4.semilogx(
        scalability_df["Demand (units/year)"],
        scalability_df["Greedy Q (units)"],
        "mo-",
        linewidth=2,
        markersize=8,
    )
    ax4.set_xlabel("Annual demand (units/year, logarithmic scale)")
    ax4.set_ylabel("Greedy order quantity Q (units)")
    ax4.set_title("Selected order quantity by demand")
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return scalability_df

scalability_results = scalability_analysis()

The instances in this experiment change demand and cost parameters, so they are separate examples rather than a controlled change in demand alone. Their timings are measurements from the current run and may vary by machine and runtime environment. The order multiple remains 50 in every instance.

In [ ]:
def advanced_constraint_analysis() -> pd.DataFrame:
    """Measure the effect of several order multiples on the base instance."""
    scenarios = [
        {"order_multiple": 1, "name": "Integer units (multiple of 1)"},
        {"order_multiple": 10, "name": "Multiples of 10"},
        {"order_multiple": 25, "name": "Multiples of 25"},
        {"order_multiple": 50, "name": "Multiples of 50 (base)"},
        {"order_multiple": 100, "name": "Multiples of 100"},
        {"order_multiple": 250, "name": "Multiples of 250"},
    ]
    results = []

    for scenario in scenarios:
        solver = GreedyEOQWithShortages(
            demand=36000,
            ordering_cost=150,
            holding_cost=8,
            shortage_cost=12,
            order_multiple=scenario["order_multiple"],
        )
        solution = solver.greedy_search()
        Q_cont, S_cont = solver.get_optimal_continuous_solution()
        continuous_cost = solver.calculate_total_cost(Q_cont, S_cont)

        results.append({
            "Constraint": scenario["name"],
            "Order multiple (units)": scenario["order_multiple"],
            "Greedy Q (units)": solution["Q_optimal"],
            "Greedy S (units)": solution["S_optimal"],
            "Greedy cost ($/year)": solution["total_cost"],
            "Continuous lower bound ($/year)": continuous_cost,
            "Gap above lower bound (%)": (
                (solution["total_cost"] - continuous_cost) / continuous_cost * 100
            ),
            "Iterations": solution["iterations"],
        })

    constraint_df = pd.DataFrame(results)
    print("=" * 120)
    print("ORDER-MULTIPLE SENSITIVITY FOR THE BASE INSTANCE")
    print(constraint_df.round(3).to_string(index=False))
    print("=" * 120)

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Search Results by Required Order Multiple", fontsize=16, fontweight="bold")

    x_values = constraint_df["Order multiple (units)"]
    ax1.plot(x_values, constraint_df["Gap above lower bound (%)"], "ro-", linewidth=2, markersize=8)
    ax1.set_xlabel("Order multiple (units)")
    ax1.set_ylabel("Gap above continuous lower bound (%)")
    ax1.set_title("Cost gap by order multiple")
    ax1.set_xscale("log")
    ax1.grid(True, alpha=0.3)

    ax2.plot(x_values, constraint_df["Greedy Q (units)"], "bo-", linewidth=2, markersize=8)
    ax2.set_xlabel("Order multiple (units)")
    ax2.set_ylabel("Greedy order quantity Q (units)")
    ax2.set_title("Order quantity by order multiple")
    ax2.set_xscale("log")
    ax2.grid(True, alpha=0.3)

    ax3.plot(x_values, constraint_df["Greedy S (units)"], "go-", linewidth=2, markersize=8)
    ax3.set_xlabel("Order multiple (units)")
    ax3.set_ylabel("Greedy shortage level S (units)")
    ax3.set_title("Shortage level by order multiple")
    ax3.set_xscale("log")
    ax3.grid(True, alpha=0.3)

    ax4.plot(x_values, constraint_df["Iterations"], "mo-", linewidth=2, markersize=8)
    ax4.set_xlabel("Order multiple (units)")
    ax4.set_ylabel("Iterations")
    ax4.set_title("Iterations by order multiple")
    ax4.set_xscale("log")
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return constraint_df

constraint_results = advanced_constraint_analysis()

## 3. Interpretation and appropriate use

The order-multiple experiment isolates the effect of changing the permitted batch size while holding the base demand and cost parameters fixed. A larger multiple restricts the set of permitted order quantities, but the heuristic's result can also depend on its starting point and tested moves. The continuous reference remains the same across these scenarios because its model parameters do not change.

### Strengths and limitations

**Strengths**

- The search explicitly enforces a multiple constraint on the order quantity.
- The cost components and decisions remain visible, making the construction easy to inspect.
- The history and experiments support analysis of the search's behavior on the stated instances.

**Limitations**

- The method tests only a specified set of local moves and has no global optimality guarantee.
- The result can depend on the starting point, step sizes, tolerance, and iteration limit.
- The model supports an order-quantity multiple, but does not implement general supplier minimums, maximums, or other constraints.
- The model assumes deterministic demand and planned backlogging; it does not represent lost sales, stochastic demand, or multi-product interactions.

This method is appropriate when a quick, interpretable feasible construction is useful and the decision-maker can evaluate the limits of a heuristic result. If a discrete global optimum is required, this notebook's greedy result alone is insufficient evidence of one.

In [ ]:
print("=" * 76)
print("EOQ WITH PLANNED SHORTAGES: RUN SUMMARY")
print("=" * 76)
print(f"Order quantity: {greedy_solution['Q_optimal']:,.0f} units")
print(f"Maximum planned backlog: {greedy_solution['S_optimal']:,.2f} units")
print(f"Annual cost: ${greedy_solution['total_cost']:,.2f}")
print(
    "Demand immediately supplied from on-hand stock: "
    f"{(1 - greedy_solution['S_optimal'] / greedy_solution['Q_optimal']) * 100:.1f}%"
)
print(f"Search iterations: {greedy_solution['iterations']}")
print(f"Stopped after a no-improvement iteration: {greedy_solution['converged']}")
print(f"Gap above continuous lower bound: {comparison_results['percentage_gap']:.4f}%")
print(f"Measured greedy-search time: {comparison_results['greedy_time_ms']:.4f} ms")
print("\nInterpretation:")
print("- The result is feasible for the stated order-multiple constraint.")
print("- The continuous optimum is a lower bound, not a discrete optimality certificate.")
print("- The sensitivity experiments describe only the instances and starts they test.")
print("=" * 76)